# Bank Customer Churn — a Comparative Study of Classical ML Models

**Task:** given a bank customer's data (age, balance, country, number of products...), predict whether they will **leave the bank** (`Exited`: 1 = churned, 0 = stayed). Binary classification on tabular data — the bread-and-butter task of DS in banking/fintech.

**Data:** [Bank Customer Churn](https://www.kaggle.com/datasets/shrutimechlearn/churn-modelling) — 10,000 customers, 14 features, ~80/20 class imbalance.

**Idea:** climb a ladder of models from simple to complex and, at each rung, understand *why* the next one is needed:

`logistic regression → decision tree → random forest → CatBoost`

**Metric:** ROC-AUC (robust to imbalance) + precision/recall on the churn class (the business cares about catching leavers). Accuracy is ignored — with an 80/20 split it is misleading.

In [1]:
import pandas as pd

df = pd.read_csv("Churn_Modelling.csv")

print("Shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
print("\nClass balance:\n", df["Exited"].value_counts(normalize=True))
df.head()

Shape: (10000, 14)

Missing values:
 RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

Class balance:
 Exited
0    0.7963
1    0.2037
Name: proportion, dtype: float64


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [2]:
# drop identifier columns — unique per customer, no predictive signal
df = df.drop(columns=["RowNumber", "CustomerId", "Surname"])

X = df.drop(columns=["Exited"])
y = df["Exited"]

# one-hot encode categoricals for logreg / sklearn trees
X = pd.get_dummies(X, columns=["Geography", "Gender"], drop_first=True)

from sklearn.model_selection import train_test_split
# stratify keeps the 80/20 class ratio in both train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape, X_test.shape)
print(X.columns.tolist())

(8000, 11) (2000, 11)
['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_Germany', 'Geography_Spain', 'Gender_Male']


## Models: from simple to complex

Each next model beats the previous one on ROC-AUC. Trees / forests / boosting need no scaling (they split on thresholds, not distances) — `StandardScaler` is only needed for logistic regression.

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

# scale features — linear models are sensitive to feature magnitude
# fit on train only, transform test (test must not leak into the scaler)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# class_weight='balanced' compensates for the 80/20 imbalance
logreg = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg.fit(X_train_s, y_train)

proba = logreg.predict_proba(X_test_s)[:, 1]

print("Logistic Regression ROC-AUC:", round(roc_auc_score(y_test, proba), 4))
print(classification_report(y_test, logreg.predict(X_test_s)))

Logistic Regression ROC-AUC: 0.7771
              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1593
           1       0.39      0.70      0.50       407

    accuracy                           0.71      2000
   macro avg       0.65      0.71      0.65      2000
weighted avg       0.80      0.71      0.74      2000



In [4]:
from sklearn.tree import DecisionTreeClassifier

# max_depth=5 limits complexity so the tree does not memorize the training set
tree = DecisionTreeClassifier(max_depth=5, random_state=42, class_weight="balanced")
tree.fit(X_train, y_train)   # trees need no scaling — they split on thresholds, not distances

proba_tree = tree.predict_proba(X_test)[:, 1]
print("Decision Tree ROC-AUC:", round(roc_auc_score(y_test, proba_tree), 4))
print(classification_report(y_test, tree.predict(X_test)))

Decision Tree ROC-AUC: 0.8379
              precision    recall  f1-score   support

           0       0.93      0.77      0.84      1593
           1       0.46      0.76      0.57       407

    accuracy                           0.77      2000
   macro avg       0.69      0.77      0.71      2000
weighted avg       0.83      0.77      0.79      2000



In [5]:
from sklearn.ensemble import RandomForestClassifier

# 300 trees, each on a random subsample; n_jobs=-1 uses all CPU cores
rf = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced", n_jobs=-1)
rf.fit(X_train, y_train)

proba_rf = rf.predict_proba(X_test)[:, 1]
print("Random Forest ROC-AUC:", round(roc_auc_score(y_test, proba_rf), 4))
print(classification_report(y_test, rf.predict(X_test)))

Random Forest ROC-AUC: 0.8566
              precision    recall  f1-score   support

           0       0.90      0.91      0.90      1593
           1       0.62      0.60      0.61       407

    accuracy                           0.84      2000
   macro avg       0.76      0.75      0.76      2000
weighted avg       0.84      0.84      0.84      2000



In [6]:
from catboost import CatBoostClassifier

# gradient boosting; categories still passed as one-hot here (see native version below)
cb = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=6,
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=0
)
cb.fit(X_train, y_train)

proba_cb = cb.predict_proba(X_test)[:, 1]
print("CatBoost ROC-AUC:", round(roc_auc_score(y_test, proba_cb), 4))
print(classification_report(y_test, cb.predict(X_test)))

CatBoost ROC-AUC: 0.855
              precision    recall  f1-score   support

           0       0.92      0.84      0.88      1593
           1       0.53      0.70      0.60       407

    accuracy                           0.81      2000
   macro avg       0.72      0.77      0.74      2000
weighted avg       0.84      0.81      0.82      2000



In [7]:
# which features drive churn (note: tree importance is biased toward continuous features)
importances = pd.Series(rf.feature_importances_, index=X_train.columns)
print(importances.sort_values(ascending=False))

Age                  0.249828
Balance              0.143913
EstimatedSalary      0.138093
CreditScore          0.133420
NumOfProducts        0.133122
Tenure               0.079993
IsActiveMember       0.035731
Geography_Germany    0.032912
Gender_Male          0.019969
HasCrCard            0.018103
Geography_Spain      0.014915
dtype: float64


## CatBoost with native categoricals

Above, CatBoost received categories via one-hot — stripped of its main strength. Here we feed `Geography`/`Gender` **directly** through `cat_features`: CatBoost encodes them with ordered target statistics (one informative number instead of many 0/1 columns, without target leakage). Everything else is identical — only the way categories are passed changes.

In [8]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

# reload and clean, but KEEP categoricals as text (no one-hot this time)
df_cb = pd.read_csv("Churn_Modelling.csv").drop(columns=["RowNumber", "CustomerId", "Surname"])
X_cb = df_cb.drop(columns=["Exited"])
y_cb = df_cb["Exited"]

cat_features = ["Geography", "Gender"]

# same split (same random_state/stratify) for a fair comparison with the forest
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_cb, y_cb, test_size=0.2, random_state=42, stratify=y_cb
)

# same hyperparameters as the one-hot CatBoost above — only the encoding changes
cb2 = CatBoostClassifier(
    iterations=300, learning_rate=0.1, depth=6, auto_class_weights="Balanced", random_seed=42, verbose=0
)
cb2.fit(Xc_train, yc_train, cat_features=cat_features)   # feed categories directly

proba_cb2 = cb2.predict_proba(Xc_test)[:, 1]
print("CatBoost + cat_features ROC-AUC:", round(roc_auc_score(yc_test, proba_cb2), 4))
print(classification_report(yc_test, cb2.predict(Xc_test)))

CatBoost + cat_features ROC-AUC: 0.8593
              precision    recall  f1-score   support

           0       0.92      0.85      0.88      1593
           1       0.54      0.71      0.61       407

    accuracy                           0.82      2000
   macro avg       0.73      0.78      0.75      2000
weighted avg       0.84      0.82      0.83      2000



## Hyperparameter tuning (GridSearchCV)

Grid search with 3-fold cross-validation over CatBoost's `depth`, `learning_rate` and `l2_leaf_reg`, optimizing ROC-AUC. Cross-validation runs on the training set only; the test set stays untouched during tuning and is used once at the end for an honest estimate.

Note: `cat_features` is passed to `.fit()`, not to the constructor — `GridSearchCV` clones the estimator, and per-fit arguments must not live in the constructor.

In [9]:
from sklearn.model_selection import GridSearchCV

# base model WITHOUT cat_features in the constructor (GridSearch clones the estimator)
base = CatBoostClassifier(iterations=200, auto_class_weights="Balanced", random_seed=42, verbose=0)

param_grid = {
    "depth": [4, 6, 8],
    "learning_rate": [0.03, 0.1],
    "l2_leaf_reg": [1, 3, 5],
}

# 3-fold CV, optimizing ROC-AUC; n_jobs=-1 evaluates candidates in parallel
grid = GridSearchCV(base, param_grid, scoring="roc_auc", cv=3, n_jobs=-1, verbose=1)
grid.fit(Xc_train, yc_train, cat_features=cat_features)   # cat_features -> fit, not constructor

print("Best params:", grid.best_params_)
print("Best CV ROC-AUC:", round(grid.best_score_, 4))

# final honest check on the held-out test set (untouched during tuning)
best = grid.best_estimator_
proba_best = best.predict_proba(Xc_test)[:, 1]
print("Test ROC-AUC:", round(roc_auc_score(yc_test, proba_best), 4))

Fitting 3 folds for each of 18 candidates, totalling 54 fits


Best params: {'depth': 6, 'l2_leaf_reg': 1, 'learning_rate': 0.03}
Best CV ROC-AUC: 0.8687
Test ROC-AUC: 0.8736


## Results

| Model | ROC-AUC | recall (churn) |
|---|:---:|:---:|
| Logistic Regression | 0.777 | 0.70 |
| Decision Tree | 0.8379 | 0.76 |
| Random Forest | 0.8566 | 0.60 |
| CatBoost (one-hot) | 0.855 | 0.70 |
| CatBoost (cat_features) | 0.8593 | 0.71 |
| **CatBoost (tuned, GridSearchCV)** | **0.8736** | — |

**Findings:**
1. Climbing the ladder `logreg → tree → forest → boosting` steadily improves ROC-AUC — non-linear ensemble models capture feature interactions a linear model cannot.
2. **CatBoost's native categorical handling (`cat_features`) beats one-hot** on the same data and the same hyperparameters (0.855 → 0.8593), with the best recall on churners. Same lesson as the theory: target encoding is more informative than splitting a category into 0/1 columns.
3. **Hyperparameter tuning adds another jump** (0.8593 → 0.8736). The winning config uses a *low* `learning_rate` (0.03): smaller boosting steps are more careful and more accurate, given enough trees.
4. The strongest churn driver by feature importance is **age**, followed by financial features (balance, salary, credit score). *Caveat:* tree feature importance is biased toward continuous features and understates one-hot categories — permutation importance / SHAP would be more reliable.

**Next steps:** decision-threshold analysis (recall vs precision), permutation importance / SHAP.